# __vlm_check — Qwen3.5 (Qwen/Qwen3.8-27B) load + qualitative sanity

Branch `vlm-deltanet`. First contact with the hybrid VLM (Gated-DeltaNet + gated full-attention).
This is a **qualitative** check, not a systematic eval:

1. does `ModelandTokenizer` load it (VLM auto-class, `text_config`, `model.language_model.*` paths)?
2. is the parsed config right — 64 layers, 16 full-attention layers, head_dim 256, o_proj at 6144?
3. does it write coherent **code** on a simple task, through *our* chat-template setup?
4. does the extraction machinery run on it — `prepare_input`, `predict_next_token`,
   `get_o_proj_inputs_batch` at a full-attention layer?

Nothing here touches the DeltaNet layers or trains anything; we only read/generate.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4,7")   # ~54GB bf16 weights -> 2x80GB or adjust
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import baukit
from src.utils import logging_utils
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)

MODEL_NAME = "Qwen/Qwen3.8-27B" 

## 1. Load

`ModelandTokenizer` now auto-detects the `*ForConditionalGeneration` checkpoint and loads it via
`AutoModelForImageTextToText` (nnsight defaults to `AutoModelForCausalLM`, which can't). The vision
tower loads but never runs for text-only. Default attention impl for now — eager is tried in §4.

In [2]:
from src.models import ModelandTokenizer

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    # attn_implementation="eager",   # enable when you need attention patterns (§4)
)
print("loaded:", mt.name)
print("model class:", type(mt._model).__name__)

2026-08-25 18:41:20 src.models WARNING  Qwen/Qwen3.8-27B not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-08-25 18:41:20 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.8-27B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-25 18:41:20 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.8-27B/1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0/config.json "HTTP/1.1 200 OK"
2026-08-25 18:41:20 src.models INFO     detected VLM (Qwen3_5ForConditionalGeneration); loading via AutoModelForImageTextToText
2026-08-25 18:41:20 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.8-27B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-25 18:41:20 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.8-27B/1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0/config.json "HTTP/1

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

2026-08-25 18:41:41 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.8-27B/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-25 18:41:41 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.8-27B/1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0/generation_config.json "HTTP/1.1 200 OK"
2026-08-25 18:41:42 src.models INFO     loaded model <Qwen/Qwen3.8-27B> | size: 52178.819 MB | dtype: torch.bfloat16 | device: cuda:0
loaded: Qwen/Qwen3.8-27B
model class: Qwen3_5ForConditionalGeneration


## 2. Parsed-config sanity

Expected: `n_layer=64`, `n_embd=5120`, `n_heads=24`, `head_dim=256`, `attn_inner_size=6144`,
`attention_layers` = 16 layers at [3, 7, …, 63], paths under `model.language_model.*`.

In [3]:
print(f"n_layer          : {mt.n_layer}")
print(f"n_embd           : {mt.n_embd}")
print(f"n_heads          : {mt.n_heads}")
print(f"head_dim         : {mt.head_dim}")
print(f"attn_inner_size  : {mt.attn_inner_size}   (o_proj input width)")
print(f"attention_layers : {len(mt.attention_layers)} -> {mt.attention_layers}")
print(f"layer_name_format: {mt.layer_name_format}")
print(f"attn_module_fmt  : {mt.attn_module_name_format}")
print(f"embedder / norm  : {mt.embedder_name} | {mt.final_layer_norm_name}")
print(f"lm_head          : {mt.lm_head_name}")

# every declared path resolves; a full-attn layer has o_proj, a linear layer does not
a = mt.attention_layers[0]
for p in [mt.embedder_name, mt.final_layer_norm_name, mt.lm_head_name,
          mt.attn_module_name_format.format(a) + ".o_proj"]:
    ok = "OK"
    try: baukit.get_module(mt._model, p)
    except Exception as e: ok = f"FAIL {e}"
    print(f"  resolve  {p:52s} {ok}")

lin = next(i for i in range(mt.n_layer) if i not in set(mt.attention_layers))
try:
    baukit.get_module(mt._model, mt.attn_module_name_format.format(lin))
    print(f"  layer {lin} self_attn: UNEXPECTEDLY present")
except Exception:
    print(f"  layer {lin} self_attn: correctly absent (linear/DeltaNet)")
    print(f"  layer {lin} linear_attn present:",
          baukit.get_module(mt._model, mt.layer_name_format.format(lin) + ".linear_attn") is not None)

n_layer          : 64
n_embd           : 5120
n_heads          : 24
head_dim         : 256
attn_inner_size  : 6144   (o_proj input width)
attention_layers : 16 -> (3, 7, 11, 15, 19, 23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63)
layer_name_format: model.language_model.layers.{}
attn_module_fmt  : model.language_model.layers.{}.self_attn
embedder / norm  : model.language_model.embed_tokens | model.language_model.norm
lm_head          : lm_head
  resolve  model.language_model.embed_tokens                    OK
  resolve  model.language_model.norm                            OK
  resolve  lm_head                                              OK
  resolve  model.language_model.layers.3.self_attn.o_proj       OK
  layer 0 self_attn: correctly absent (linear/DeltaNet)
  layer 0 linear_attn present: True


## 3. Does it write code?

Through *our* chat-template path (`src.goals.functional.apply_chat_template`, `<code>` tags,
`enable_thinking=False`). Greedy `generate_with_patch` — this also exercises the manual position
handling (M-RoPE degenerates to 1D for text; if the output is coherent, positions are fine).

In [4]:
from src.goals.functional import apply_chat_template
from src.functional import generate_with_patch

instruction = (
    "Write a Python function `merge_sort(arr: list) -> list` that sorts an array using the merge sort algorithm. "
    "Wrap the code in <code></code>. Do not include comments or explanations."
)
prompt = apply_chat_template(mt=mt, user_query=instruction, incomplete_code="<code>\n")
gen = generate_with_patch(mt=mt, inputs=prompt, max_new_tokens=200,
                          do_sample=False, remove_prefix=True)[0]
print(gen)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.



def merge_sort(arr: list) -> list:
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left: list, right: list) -> list:
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result
</code>
</code>


In [5]:
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape
import logging
from src.utils import logging_utils

console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)

In [6]:
from src.goals.multiline_prompts import (
    build_example,
    replace_in_code_newline_with_tabnewline,
)
from src.functional import generate_with_patch
import random

#####################################################################
NAMING = "letters"  # "letters" | "natural" | "indexed"
N_GEN = 32
#####################################################################

seed = random.Random(234)
souce_example = build_example(
    mt=mt,
    rng=seed,
    target_slot=1,
    n_conditions=4,
    mode="chat",
    # style="singleline" if ss=="sl" else "multiline",
    style="multiline",
    naming=NAMING,
)

src_prompt = souce_example["full_prompt"]

prefix="\n"

src_prompt = src_prompt + prefix
src_prompt = replace_in_code_newline_with_tabnewline(src_prompt) + "   "

# src_prompt = "Here's a code to implement bubble sort. \n<code>\n"

source_gen = generate_with_patch(
    mt,
    inputs=src_prompt,
    n_gen_per_prompt=1,
    max_new_tokens=N_GEN,
    do_sample=False,
    remove_prefix=True,
)[0]
rich_print(
    f"[white]{escape(src_prompt)}[/white][medium_orchid1]{escape(source_gen)}[/medium_orchid1]"
)

<|im_start|>system
Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider 
plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.

You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n is a power of two
2. n has at least 4 digits
3. n is odd
4. the sum of n's digits is divisible by 9

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using 
single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all 
conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<think>

</think>

<code>    
def check_conditions(n: int):    
    a = bin(n).count("1")    
    a = n > 0 and a == 1    
    b = len(str(abs(n)))    
    b = b >= 4    
    c = n % 2    
    c = c == 1    
    d = sum(int(ch) for ch

## 4. Our extraction machinery on this arch

`prepare_input` → `predict_next_token` (next-token top-k), then `get_o_proj_inputs_batch` at the
full-attention layers only. Expected o_proj-input width per layer = `attn_inner_size` (6144), i.e.
gated per-head attention outputs concatenated **before** o_proj.

In [7]:
from src.tokens import prepare_input
from src.functional import predict_next_token
from src.goals.head_loc_attn import get_o_proj_inputs_batch

inp = prepare_input(prompts=prompt, tokenizer=mt)
print("input_ids:", tuple(inp.input_ids.shape))

topk = predict_next_token(mt=mt, inputs=inp, k=8)
print("next-token top-8:", [(t.token, round(t.prob, 3)) for t in topk[0]])

# o_proj inputs at the full-attention layers (must use attention_layers, not range(n_layer))
layers = list(mt.attention_layers)
o = get_o_proj_inputs_batch(mt, inp, layers, token_positions=[-1])
L0 = layers[0]
print(f"o_proj input @ L{L0}: {tuple(o[L0].shape)}  (expect (batch, 1, {mt.attn_inner_size}))")
print("captured layers:", list(o.keys()))

input_ids: (1, 138)
next-token top-8: [('\n', 0.922), ('def', 0.036), ('\n\n', 0.031), ('python', 0.005), ('\n\n\n', 0.001), ('merge', 0.001), ('"""', 0.001), ('```', 0.001)]
o_proj input @ L3: (1, 1, 6144)  (expect (batch, 1, 6144))
captured layers: [3, 7, 11, 15, 19, 23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63]


## 5. Intervention correctness — the output gate is BEFORE `o_proj`

`Qwen3_5Attention.forward` ends with:

```python
attn_output = attn_output.reshape(*input_shape, -1)   # (b, s, n_heads*head_dim) = 6144
attn_output = attn_output * torch.sigmoid(gate)       # gate = 2nd half of q_proj, per head-element
attn_output = self.o_proj(attn_output)                # o_proj consumes the GATED output
return attn_output, attn_weights                      # <- added to the residual by the decoder layer
```

So **`o_proj.input == gate ⊙ OV`** — exactly what the head writes — and since `o_proj` is bias-free
(`attention_bias=false`), the sum of per-head writes `o_proj_input[:, h] @ W_oᵀ[h]` equals the
attention block's residual write. Every o_proj-input intervention we have
(`collect_head_steering_vecs`, `src_replacement_hooks`, replace-strategy `apply_goalscope`) is valid
**unchanged** — the per-head math already keys off `mt.n_heads` (24) / `mt.head_dim` (256) /
`mt.attn_inner_size` (6144). Two checks confirm it, using the *actual pipeline functions*.

### 5a. Reconstruction — sum of per-head writes == the attention block's residual write

`collect_head_steering_vecs` over ALL heads at a full-attention layer must reproduce that layer's
`self_attn` module output (up to bf16). If it does, gating is pre-o_proj and the decomposition is
exact.

In [8]:
from src.goals.head_loc_attn import collect_head_steering_vecs

ATTN = list(mt.attention_layers)
L = ATTN[len(ATTN) // 2]
T = inp.input_ids.shape[1]
positions = list(range(T - 4, T))                      # last 4 tokens

# ground truth: the attn block's residual write (its module output) at those positions
attn_name = mt.attn_module_name_format.format(L)
with torch.no_grad(), baukit.TraceDict(mt._model, [attn_name], retain_output=True) as td:
    mt._model(input_ids=inp.input_ids, attention_mask=inp.attention_mask)
y = td[attn_name].output
y = (y[0] if isinstance(y, tuple) else y)[0, positions, :].float().cpu()   # (n_pos, n_embd)

# pipeline: sum of ALL heads' writes at L  (== o_proj(o_proj_input), bias-free)
all_heads = [(L, h) for h in range(mt.n_heads)]
recon = collect_head_steering_vecs(mt, inp, all_heads, positions)[0]       # (n_pos, n_embd)

diff = (recon - y).abs().max().item()
cos = torch.nn.functional.cosine_similarity(recon, y, dim=-1).mean().item()
print(f"L{L}: max|recon - attn_out| = {diff:.4g}   mean cos = {cos:.6f}")
print("OK: gate is pre-o_proj; per-head OV reconstructs the residual write"
      if cos > 0.999 else "MISMATCH: investigate gate placement / head slicing")

L35: max|recon - attn_out| = 0.00314   mean cos = 0.999999
OK: gate is pre-o_proj; per-head OV reconstructs the residual write


### 5b. Head replacement fires on the gated-attention layers

`forward_with_src_replacement` transplants source o_proj inputs at a pivot for a set of heads (over
`mt.attention_layers`) and must change the logits. Clean = the same call with a zero mask (a no-op),
so any delta is the transplant.

In [9]:
from src.goals.head_loc_attn import HeadMask, forward_with_src_replacement

# source o_proj inputs at the last token of a DIFFERENT prompt
src_prompt = apply_chat_template(mt=mt, user_query="Write a function that returns the sum of a list.",
                                 incomplete_code="<code>\n")
src_inp = prepare_input(prompts=src_prompt, tokenizer=mt)
src_o = get_o_proj_inputs_batch(mt, src_inp, ATTN, token_positions=[-1])
src_stack = torch.stack([src_o[l][0, 0] for l in ATTN])             # (n_attn, 6144) float32

# mask: all heads ON across a mid band of attention layers
mask = HeadMask(len(ATTN), mt.n_heads); mask_zero = HeadMask(len(ATTN), mt.n_heads)
with torch.no_grad():
    mask.values.zero_(); mask_zero.values.zero_()
    mid = len(ATTN) // 2
    mask.values[mid - 2:mid + 2] = 1.0

pos = [T - 1]
clean   = forward_with_src_replacement(mt, dict(inp), src_stack, mask_zero, mask_layers=ATTN, patch_positions=pos, alpha=1.0)
patched = forward_with_src_replacement(mt, dict(inp), src_stack, mask,      mask_layers=ATTN, patch_positions=pos, alpha=1.0)

d = (patched - clean).abs().max().item()
tc = mt.tokenizer.decode(int(clean.argmax())); tp = mt.tokenizer.decode(int(patched.argmax()))
n_layers_active = int((mask.values.sum(1) > 0).sum())
print(f"replace: {int(mask.n_active())} heads across {n_layers_active} full-attn layers")
print(f"max|Δlogit| = {d:.4g}   top clean={tc!r}   top patched={tp!r}")
print("OK: o_proj-input replacement hooks fire on gated-attention layers"
      if d > 0 else "NO EFFECT: hooks not firing")

replace: 96 heads across 4 full-attn layers
max|Δlogit| = 0.4375   top clean='\n'   top patched='\n'
OK: o_proj-input replacement hooks fire on gated-attention layers


## 6. Goal-head localization readiness

Localization (`train_head_mask`) needs two things to hold on this hybrid arch:

1. the layer set is `mt.attention_layers` (the 16 full-attn layers) — `scripts/train_head_mask.py`
   now restricts its `--layers-start..end` band to those automatically (no-op on homogeneous models);
2. **autograd flows through the DeltaNet layers to the mask** — the mask sits at attention-layer
   o_proj inputs, but the KL loss is on the final logits, so gradient must traverse the linear/
   DeltaNet layers in between. The FLA fast-path isn't installed (pure-torch fallback), which should
   be differentiable — §6a checks it directly, using the *same* `forward_with_src_replacement` the
   training inner loop backprops through.

`apply_goalscope` needs no change: its additive path injects into the residual via
`layer_name_format` (any layer) and its source/replace paths run over the selected heads' attention
layers. §6b runs it end-to-end on an arbitrary head set to confirm the tool works on this model.

### 6a. Backward through the hybrid stack (the #6 risk)

Short prompt to keep activation memory modest. If this OOMs or errors, learned-mask localization
needs grad-checkpointing (or an inference-only scorer); if `grad` is finite and non-zero, we're
clear to train a goal mask over `mt.attention_layers`.

In [ ]:
from src.goals.head_loc_attn import HeadMask, forward_with_src_replacement

short = prepare_input(
    prompts=apply_chat_template(mt=mt, user_query="Return the number one.",
                                incomplete_code="<code>\ndef f():\n    return "),
    tokenizer=mt,
)
Ts = short.input_ids.shape[1]

mask_g = HeadMask(len(ATTN), mt.n_heads)          # ones, requires_grad=True (same as train init)
logits = forward_with_src_replacement(
    mt, dict(short), src_stack, mask_g, mask_layers=ATTN, patch_positions=[Ts - 1], alpha=1.0,
)                                                  # (vocab,), differentiable w.r.t. mask_g.values
target = int(logits.argsort()[-2])                 # a non-argmax target -> clearly non-zero gradient
loss = torch.nn.functional.cross_entropy(logits.unsqueeze(0),
                                         torch.tensor([target], device=logits.device))
loss.backward()

g = mask_g.values.grad
print(f"loss = {loss.item():.4f}")
print(f"grad: finite={bool(torch.isfinite(g).all())}  nonzero={int((g != 0).sum())}/{g.numel()}  "
      f"|grad|max={g.abs().max().item():.3e}")
print("OK: autograd flows through the DeltaNet layers -> learned goal-mask localization is feasible"
      if torch.isfinite(g).all() and (g != 0).any()
      else "BACKWARD ISSUE: need grad-checkpointing or an inference-only localizer")

### 6b. `apply_goalscope` end-to-end on the VLM

Arbitrary attention-layer heads (no trained mask yet) — the read-outs are not meaningful, this only
confirms the tool runs on this arch and returns one verbalization per scoped token.

In [ ]:
from src.goals.goalscope import apply_goalscope

gs_prompt = apply_chat_template(
    mt=mt, user_query="Write is_even(n).",
    incomplete_code="<code>\ndef is_even(n):\n    return n % 2 == 0\n</code>",
)
sel = sorted((L, h) for L in ATTN[6:10] for h in range(4))     # 16 arbitrary full-attn heads

res = apply_goalscope(
    gs_prompt, mt=mt, selected_heads=sel,
    strategy="additive", alpha=0.2, steer_layer_range=(35, 46),
    batch_size=8, max_new_tokens=24, show=False,
)
print(f"apply_goalscope: {len(res)} read-outs over {len(sel)} heads (layers {sorted({l for l,_ in sel})})")
for r in res[:5]:
    print(f"  tok {r['token']!r:>10}  ->  {r['patched_gen'][:70]!r}")

### Notes / next

- If §3 is coherent and §4 shapes are right, the load + config + o_proj extraction path is sound on
  this arch.
- Not yet done (later, per the plan): thread `mt.attention_layers` through the head-mask / grids;
  gate-placement numerical check (sum of per-head OV writes reconstructs the block's residual write);
  decode-parity vs `.generate()`; decide learned-vs-inference localization.
- To reclaim VRAM you can `del mt._model.model.visual` after load (text-only never uses it) — try
  once generation is confirmed working.